#  Model Training


## Generate data

In [1]:
import agama
import torch 
import numpy as np
from astropy import units as u

from sbi.utils import BoxUniform
from sbi.inference import SNLE, simulate_for_sbi, prepare_for_sbi

from sklearn.metrics import mean_squared_error, r2_score

import pandas as pd


In [2]:
# set agama unit to be in Msun, kpc, km/s
agama.setUnits(mass=1 * u.Msun, length=1*u.kpc, velocity=1 * u.km /u.s)

In [3]:
def make_potential(p_0: float, r_s: float, gamma: float) -> agama.Potential:
    """
    Makes potential according to GNFW profile
    
    - p_0: density normalization
    - r_s: scale radius
    - gamma: inner slope

    """

    # Based on GNFW Profile
    param = {
        "type": "Spheroid", 
        "densityNorm": p_0,
        "scaleRadius": r_s,
        "gamma": gamma,
        "beta": 3,
        "alpha": 1
    }

    return agama.Potential(param)

def make_density(r_star: float):
    """
    Creates stellar density distribution according to the 3D Plummer Profile

    - r_star: scale length
    """

    # Based on Plummer profile
    param = {
        "type": "Plummer",
        "mass": 1,
        "scaleRadius": r_star,
    }
    
    return agama.Density(param)


def generate_galaxy(p_0: float, r_s: float, gamma: float, r_star: float):
    """
    Generate the galaxy model given theta

    - p_0: density normalization
    - r_s: scale radius
    - gamma: inner slope
    - r_star: scale length

    """

    pot = make_potential(p_0, r_s, gamma)

    df = agama.DistributionFunction(
        type = "QuasiSpherical",
        potential = pot,
        density = make_density(r_star),
        beta0 = 0.0,
        r_a = np.inf
    )

    return agama.GalaxyModel(pot, df)

def transform_params(theta: torch.Tensor) -> torch.Tensor:
    """
    transform parameters into correct for generate_galaxy

    - theta: tensor of sampled theta with columns \
        log(p_0), log(r_s), gamma, r_star/r_s
    """


    p_0 = 10 ** theta[:,0]
    r_s = 10 ** theta[:,1]
    gamma = theta[:,2]
    r_star = theta[:,3] * r_s

    return torch.stack([p_0, r_s, gamma, r_star], dim=1)



def generate_galaxy_multiple(theta: torch.Tensor, n_stars:int = 1) -> torch.Tensor:
    """
    Generate the galaxy model with multiple stars given theta 

    returns a matrix of stars for each theta

    - theta: tensor of sampled theta with columns \
        log(p_0), log(r_s), gamma, r_star/r_s
    """
    transformed_theta = transform_params(theta)
    
    samples_np = np.vstack([[generate_galaxy(*row.tolist()).sample(n_stars)[0]] for row in transformed_theta])

    return torch.from_numpy(samples_np).to(torch.float32)  # sbi requires float 32

## Training

In [20]:
agama.setRandomSeed(13)
torch.manual_seed(13)
np.random.seed(13)

### Generating data

In [5]:
def generate_prior():
    """
    Generate the prior assuming the uniform distribution
    """
    

    low = torch.tensor([5, -1, -1, 0.2])
    high = torch.tensor([8, 0.7, 2, 1])

    # Create uniform distribution
    prior = BoxUniform(low=low, high=high)

    return prior

In [6]:
prior = generate_prior()
simulator_sbi, prior_sbi = prepare_for_sbi(generate_galaxy_multiple, prior)

In [13]:
#. Generate Training Data
theta_train, x_train = simulate_for_sbi(
    simulator_sbi, prior_sbi, num_simulations=1000, num_workers=1
)
x_train = x_train.squeeze()

Running 1000 simulations.:   0%|          | 0/1000 [00:00<?, ?it/s]

#### Generating 10000 data then saving to csv

In [7]:
prior = generate_prior()

In [51]:
theta = prior.sample((10000,))
df = pd.DataFrame(np.repeat(theta, 100, axis=0))
df.to_csv("./data/training_theta.csv", index=False, header=False)

In [52]:
x = generate_galaxy_multiple(theta, 100).numpy()
out = x.reshape(x.shape[0] * x.shape[1], x.shape[2])
df = pd.DataFrame(out)

df.to_csv("./data/training_x.csv", index=False, header=False)

### Train the model

In [53]:
theta_train = torch.from_numpy(np.array(pd.read_csv("./data/training_theta.csv", header=None))).float()
x_train = torch.from_numpy(np.array(pd.read_csv("./data/training_x.csv", header=None))).float()

In [56]:
prior_sbi = generate_prior()
inference = SNLE(prior=prior_sbi)
inference.append_simulations(theta_train, x_train)

arg = {
        "training_batch_size": 200,
        "learning_rate": 0.01,
        "validation_fraction": 0.1,
        "stop_after_epochs": 10,
        "max_num_epochs": 2^31 - 1,
        "clip_max_norm": 5.0,
        "resume_training": False,
        "discard_prior_samples": False,
        "retrain_from_scratch": False,
        "show_train_summary": True,
        "dataloader_kwargs": None,
}

likelihood_estimator = inference.train(**arg)

KeyboardInterrupt: 

In [77]:
posterior = inference.build_posterior(likelihood_estimator, mcmc_method="slice_np_vectorized")

In [78]:
theta_test = prior.sample((1,))
x_test = generate_galaxy_multiple(theta_test, 1).squeeze()
samples = posterior.sample((1000,), x=x_test)


Running vectorized MCMC with 1 chains:   0%|          | 0/10100 [00:00<?, ?it/s]

In [1]:
print(np.median(samples, axis=0))
print(theta_test)

NameError: name 'np' is not defined

## Evaluating model performance

In [ ]:
def evaluate_model(theta, medians):
    for i in range(5):
        mse = mean_squared_error(theta[:,i], medians[:,i])
        r2 = r2_score(theta[:,i], medians[:,i])
        print(f"MSE for {i}th entry is {mse}")
        print(f"R^2 for {i}th entry is {r2}")
        print()
